# 01 — RETAIN Model Deep Dive

RETAIN (Reverse Time Attention Network) — Choi et al. 2016.

Key differentiator: **interpretable** via dual attention mechanisms:
- **alpha weights**: visit-level attention (which visits mattered most)
- **beta weights**: code-level attention (which diagnoses/drugs mattered within each visit)

Ideal for clinical settings where explainability is required.

In [ ]:
from pyhealth_enterprise.datasets.synthetic import SyntheticEHRDataset
from pyhealth.tasks import readmission_prediction_mimic3_fn
from pyhealth.datasets import split_by_patient, get_dataloader
from pyhealth.models import RETAIN
from pyhealth.trainer import Trainer

ds = SyntheticEHRDataset(); ds.load()
task_dataset = ds.dataset.set_task(readmission_prediction_mimic3_fn)
train, val, test = split_by_patient(task_dataset, [0.8, 0.1, 0.1])
train_loader = get_dataloader(train, batch_size=32, shuffle=True)
val_loader   = get_dataloader(val,   batch_size=32, shuffle=False)
test_loader  = get_dataloader(test,  batch_size=32, shuffle=False)

model = RETAIN(
    dataset=task_dataset,
    feature_keys=['conditions', 'drugs'],
    label_key='readmission',
    mode='binary',
    embedding_dim=128,
    dropout=0.5,
)
print(model)

In [ ]:
trainer = Trainer(model=model, metrics=['pr_auc', 'roc_auc', 'f1'])
trainer.train(train_dataloader=train_loader, val_dataloader=val_loader,
              epochs=50, monitor='pr_auc')
result = trainer.evaluate(test_loader)
print('Test metrics:', result)